# PhysIR — per-image eval for the seed-1337 Colab pair (Kaggle, eval only)

`abl_baseline_s1337` and `abl_full_s1337` were **trained on Colab (L4)** to 200/200, but their
per-image evaluation was never run — only the training logs came back. This notebook evaluates
those two checkpoints and nothing else. **No training.**

**Why it matters:** the deferred seed analysis (Ch.4 guide §M.5) turns on the interaction
`full+percep − full` at seed 1337. `abl_full_percep_s1337` is already evaluated; without
`abl_full_s1337` the contrast **cannot be computed at all**, and the 1–1 split between seeds 42
and 2024 stays unbroken. `abl_baseline_s1337` completes the third replicate of
`full − baseline` (already significant at the other two seeds), so it is worth the extra ~20 min
but is not load-bearing.

## Inputs to attach
| # | dataset | note |
|---|---|---|
| 1 | `physir-code` | must contain the `shuffle=False` fix — the guard enforces it |
| 2 | `lolblur-dataset` | the 1800-image test set |
| 3 | `gabrielafotey/physir-models-evaluations-set` | the two s1337 checkpoints (`abl_baseline_s1337_latest.pt`, `abl_full_s1337_latest.pt`) were appended to this existing dataset — no separate upload needed. Nesting does not matter: checkpoints are located by filename across every attached input. |

**Accelerator: GPU T4 × 2 — strongly recommended.** One evaluation remains (`abl_full_s1337`): ~15–30 min on GPU versus ~3–6 h on CPU. `abl_baseline_s1337` is already complete and has been removed from the target list. Idempotent — an existing CSV is
skipped, so a cut session resumes cleanly.

Outputs land in `per_image/` with the same schema as every other corrected CSV
(`idx, fname, psnr, ssim, lpips, lpips_darkirconv`), clip-qualified `fname` included, so they
drop straight into the cross-seed analysis.

## 1. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops tabulate 2>/dev/null
import importlib.util as _iu
_missing = [m for m in ('ptflops','einops','pytorch_msssim','lpips','torch','torchvision','numpy')
            if _iu.find_spec(m) is None]
print('dependency check:', 'all present ✅' if not _missing else f'MISSING {_missing} ❌')
assert not _missing, f'install failed for {_missing} (Internet must be ON)'

## 2. Paths + hard guard (refuses stale physir-code)

In [ ]:
import os, sys, glob, re, json, shutil
import numpy as np, torch

INPUT_BASE = '/kaggle/input'
WORK = '/kaggle/working'
PERIMG = os.path.join(WORK, 'per_image');           os.makedirs(PERIMG, exist_ok=True)
STRIP  = os.path.join(WORK, 'checkpoints_stripped'); os.makedirs(STRIP, exist_ok=True)

ROOTS = sorted(d for d in glob.glob(os.path.join(INPUT_BASE,'*')) if os.path.isdir(d))
def _dir_containing(marker, hint=None):
    ordered = sorted(ROOTS, key=lambda d: (hint or '') not in os.path.basename(d))
    for r in ordered:
        if os.path.exists(os.path.join(r, marker)): return r
        hits = sorted(glob.glob(os.path.join(r,'**',marker), recursive=True))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep) or r
    return None

CODE_ROOT = _dir_containing('train.py', 'physir-code')
LOLBLUR   = _dir_containing(os.path.join('test','low_blur_noise'), 'lolblur')
assert CODE_ROOT and LOLBLUR, f'code/data not found among {ROOTS}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── HARD GUARD: the uploaded loader must NOT shuffle the test set ─────────────
_ds = open(os.path.join(CODE_ROOT,'data','dataset_reader','dataset_LOLBlur.py')).read()
_bad = [d for d in re.findall(r'DataLoader\([^)]*\)', _ds, re.S)
        if 'shuffle=True' in d.replace(' ', '')]
assert not _bad, ('STALE physir-code: the test DataLoader still has shuffle=True. '
                  'Re-upload physir-code with the shuffle=False fix before running.')
print('✅ guard passed — uploaded loader does not shuffle the test set')

def find_ckpt(fname):
    # Checkpoints live in `physir-models-evaluations-set` (the s1337 pair was appended to it),
    # but the search is name-based across every attached input at any depth, so the dataset
    # layout is irrelevant — nothing breaks if files move or the slug changes.
    ordered = sorted(ROOTS, key=lambda r: 'models' not in os.path.basename(r).lower())
    for r in ordered:
        for h in sorted(glob.glob(os.path.join(r, '**', fname), recursive=True)):
            if h.endswith('.pt'): return h
    return None
print('attached:', [os.path.basename(r) for r in ROOTS])

## 2b. Session restore (attach the previous version's output to resume)

In [ ]:
# ── Session restore: carry finished / partly-finished CSVs back into the working dir ──
# Continuation sessions: Add Input → Your Work → this notebook (previous version's output).
# Completed arms are then skipped; a partly-written CSV resumes mid-arm (see the eval loop).
import shutil
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/per_image/*.csv', '/kaggle/input/*/*/per_image/*.csv'):
        for src in glob.glob(pat):
            dst = os.path.join(PERIMG, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst); n += 1
    if n:
        print(f'restored {n} CSV(s) from a previous session:')
        for f in sorted(os.listdir(PERIMG)):
            rows = sum(1 for _ in open(os.path.join(PERIMG, f))) - 1
            print(f'    {f}  {rows}/1800 rows' + ('  (complete)' if rows >= 1800 else '  → will resume'))
    else:
        print('fresh session — nothing to restore')
_restore()

## 3. Targets + checkpoint availability

In [ ]:
# name → checkpoint filename.  ALIASES keep older downstream code working unchanged.
TARGETS = [
  # abl_baseline_s1337 is ALREADY DONE (a full CPU session, 1800 rows, verified) — it is
  # deliberately not listed, so it cannot be recomputed even if the restore step is skipped
  # or the previous output isn't attached. Re-add the line only if that CSV is ever lost.
  # ('abl_baseline_s1337', 'abl_baseline_s1337_latest.pt'),
  ('abl_full_s1337',     'abl_full_s1337_latest.pt'),
]

ALIASES = {}                      # no aliases needed for the replicate arms

print(f'{"target":24s} {"checkpoint":34s} status')
missing = []
for name, fn in TARGETS:
    p = find_ckpt(fn)
    out = os.path.join(PERIMG, f'per_image_{name}.csv')
    st = 'DONE ✅' if os.path.exists(out) else (
        'found: ' + os.path.relpath(p, INPUT_BASE) if p else 'CKPT MISSING ❌')
    if not p and not os.path.exists(out): missing.append(fn)
    print(f'{name:24s} {fn:34s} {st}')
if missing:
    print('\n⚠ missing checkpoints — attach the matching set-notebook output:')
    for m in missing: print('   ', m)
print(f'\n{len(TARGETS)} targets · aliases: {ALIASES}')

## 4. Eval engine (batched, per-sample metrics, filename column)

In [ ]:
import csv, time
import train as T
from data import create_test_data
from pytorch_msssim import ssim as _ssim
import lpips as _lpips_mod

BATCH = 4          # uniform 1120x640 test images → batching is safe and ~3x faster
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# The DarkIR-convention LPIPS column exists only for the cross-paper comparison; the seed
# replicates never use it. Skipping it on CPU halves the dominant cost (two VGG passes -> one).
DUAL_LPIPS = (dev == 'cuda')
print('device:', dev, '| batch', BATCH,
      '| dual-LPIPS', 'on' if DUAL_LPIPS else 'OFF (CPU: not needed for replicate arms)')
if dev != 'cuda':
    torch.set_num_threads(os.cpu_count() or 4)
    print(f'  CPU run: {os.cpu_count()} threads. Expect ~3-6 h per arm; the loop is idempotent,')
    print('  so a cut session resumes from the next unfinished CSV.')

opt_test = {'name':'LOLBlur',
            'val':{'test_path': os.path.join(LOLBLUR,'test'), 'batch_size_test': BATCH},
            'train':{'n_workers':2, 'verbose':False}}
raw,_ = create_test_data(rank=0, world_size=1, opt=opt_test)
LOADER = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']
DS = LOADER.dataset
# clip-qualified so the key is UNIQUE: LOLBlur test basenames repeat across its 30 clips
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p))
          for p in DS.imgs_low]                          # dataset sorts its lists
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
print(f'test set: {len(DS)} images | first: {FNAMES[0]} | last: {FNAMES[-1]}')

# runtime proof that iteration order == sorted order (i.e. idx is a stable image id)
_h, _l = next(iter(LOADER))
assert len(FNAMES) == len(DS), 'filename list length mismatch'
print('✅ order check: loader is non-shuffled, idx ↔ sorted filename is stable')

LPIPS_NET = _lpips_mod.LPIPS(net='vgg', verbose=False).to(dev).eval()
NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
       'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

def load_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
    assert st is not None, f'no weights in {path}'
    m = T.build_model(NET).to(dev)
    m.load_state_dict({k.replace('module.',''):v for k,v in st.items()}, strict=False)
    return m.eval(), ck

HEADER = ['idx', 'fname', 'psnr', 'ssim', 'lpips', 'lpips_darkirconv']

def _rows_done(path):
    """Completed rows, truncated to a whole-batch boundary so a half-written batch is redone."""
    if not os.path.exists(path): return 0
    rows = list(csv.reader(open(path)))[1:]
    done = (len(rows) // BATCH) * BATCH
    if done != len(rows):                      # drop the partial batch
        with open(path, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(HEADER); w.writerows(rows[:done])
    return done

@torch.no_grad()
def eval_model(model, out_csv):
    """Writes incrementally and resumes mid-arm, so an interrupted session loses at most
    the last few hundred images rather than the whole evaluation."""
    done = _rows_done(out_csv)
    if done: print(f'    resuming at {done}/{len(DS)}')
    fh = open(out_csv, 'a' if done else 'w', newline='')
    w = csv.writer(fh)
    if not done: w.writerow(HEADER); fh.flush()
    i0, t0 = 0, time.time()
    for high, low in LOADER:
        n = high.shape[0]
        if i0 + n <= done:                     # already computed in an earlier session
            i0 += n; continue
        high, low = high.to(dev, non_blocking=True), low.to(dev, non_blocking=True)
        pred = model(low, side_loss=False).clamp(0, 1)[:, :, :high.shape[2], :high.shape[3]]
        mse  = ((pred - high) ** 2).mean(dim=(1, 2, 3))
        psnr = 20 * torch.log10(1.0 / (mse.sqrt() + 1e-8))
        ssim = _ssim(pred, high, data_range=1.0, size_average=False)
        lp   = LPIPS_NET(pred * 2 - 1, high * 2 - 1).flatten()
        lpdk = (LPIPS_NET(pred, high).flatten() if DUAL_LPIPS
                else torch.full_like(lp, float('nan')))
        for k in range(n):
            w.writerow([i0 + k, FNAMES[i0 + k], float(psnr[k]), float(ssim[k]),
                        float(lp[k]), float(lpdk[k])])
        i0 += n
        if i0 % 200 < BATCH:
            fh.flush(); os.fsync(fh.fileno())   # survive an abrupt session kill
            r = (time.time() - t0) / max(i0 - done, 1)
            print(f'    {i0}/{len(DS)}  ~{r * (len(DS) - i0) / 60:.1f} min left', flush=True)
    fh.close()
    vals = np.array([[float(x) for x in r[2:]] for r in list(csv.reader(open(out_csv)))[1:]])
    return np.nanmean(vals, axis=0), len(vals)


## 5. RUN — 12 evals, idempotent

In [ ]:
for name, fn in TARGETS:
    out = os.path.join(PERIMG, f'per_image_{name}.csv')
    if _rows_done(out) >= len(DS):
        print(f'✅ {name}: complete ({len(DS)} rows) — skip'); continue
    p = find_ckpt(fn)
    if not p:
        print(f'⏭  {name}: checkpoint {fn} not attached — skip'); continue
    print(f'\n=== EVAL {name}  ({os.path.relpath(p, INPUT_BASE)}) ===', flush=True)
    model, ck = load_model(p)
    (ps, ss, lp, lpdk), n = eval_model(model, out)
    print(f'  → PSNR {ps:.4f}  SSIM {ss:.4f}  LPIPS {lp:.4f}  LPIPS(DarkIR-conv) {lpdk:.4f}  n={n}')
    # curated EMA-only copy for future sessions (~13 MB vs ~150 MB)
    sp = os.path.join(STRIP, os.path.basename(p))
    if not os.path.exists(sp):
        st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
        torch.save({'ema': {k.replace('module.',''):v for k,v in st.items()},
                    'epoch': ck.get('epoch') if isinstance(ck, dict) else None}, sp)
    del model; torch.cuda.empty_cache()

# alias copies so existing notebooks/table code find the names they expect
for src_name, alist in ALIASES.items():
    s = os.path.join(PERIMG, f'per_image_{src_name}.csv')
    if os.path.exists(s):
        for a in alist:
            d = os.path.join(PERIMG, f'per_image_{a}.csv')
            if not os.path.exists(d): shutil.copy2(s, d); print('alias:', os.path.basename(d))
print('\nRUN loop complete.')

## 6. Verify pairing is now real + bundle

In [ ]:
# ── The check that would have caught the original bug ────────────────────────
import csv
def load(name):
    p = os.path.join(PERIMG, f'per_image_{name}.csv')
    if not os.path.exists(p): return None
    rows = list(csv.DictReader(open(p)))
    return ([r['fname'] for r in rows], np.array([float(r['psnr']) for r in rows]))

pairs = [('stage4_physir_final','stage5_baseline'), ('stage4_physir_final','darkir_m_released'),
         ('abl_full_last','abl_baseline_last'), ('abl_full_last','abl_percep_last')]
print(f'{"pair":52s} {"fname match":>12s} {"corr":>7s}  verdict')
for a,b in pairs:
    A, B = load(a), load(b)
    if not (A and B): print(f'{a+" vs "+b:52s}  (not both present)'); continue
    same = A[0] == B[0]
    r = float(np.corrcoef(A[1], B[1])[0,1])
    v = 'PAIRED ✅' if same and r > 0.9 else 'PROBLEM ❌'
    print(f'{a+" vs "+b:52s} {str(same):>12s} {r:7.3f}  {v}')
print('\nExpect fname match = True and corr > 0.9 (image difficulty dominates and now cancels).')
print('Before the fix these correlations were ~0.0 — that was the tell.')

shutil.make_archive(os.path.join(WORK,'per_image_corrected'), 'zip', WORK, 'per_image')
print('\nper_image/:'); [print('  ', f) for f in sorted(os.listdir(PERIMG))]
print('stripped ckpts:', len(os.listdir(STRIP)), 'files')
print('\nSave a version, then attach this notebook\'s output to the figures/tables notebook')
print('(name it with "corrected" in the slug so it takes precedence).')